In [1]:
import pandas as pd
import numpy as np
df = pd.read_csv("data/raw/train.csv")

print("Размер датасета:", df.shape)
df.head()
df.info()

Размер датасета: (891, 12)
<class 'pandas.DataFrame'>
RangeIndex: 891 entries, 0 to 890
Data columns (total 12 columns):
 #   Column       Non-Null Count  Dtype  
---  ------       --------------  -----  
 0   PassengerId  891 non-null    int64  
 1   Survived     891 non-null    int64  
 2   Pclass       891 non-null    int64  
 3   Name         891 non-null    str    
 4   Sex          891 non-null    str    
 5   Age          714 non-null    float64
 6   SibSp        891 non-null    int64  
 7   Parch        891 non-null    int64  
 8   Ticket       891 non-null    str    
 9   Fare         891 non-null    float64
 10  Cabin        204 non-null    str    
 11  Embarked     889 non-null    str    
dtypes: float64(2), int64(5), str(5)
memory usage: 83.7 KB


## Первичный анализ данных

In [2]:
missing_values = df.isnull().sum()

print("Количество пропущенных значений:")
print(missing_values)

Количество пропущенных значений:
PassengerId      0
Survived         0
Pclass           0
Name             0
Sex              0
Age            177
SibSp            0
Parch            0
Ticket           0
Fare             0
Cabin          687
Embarked         2
dtype: int64


In [3]:
missing_only = missing_values[missing_values > 0]

print("Столбцы с пропусками:")
print(missing_only)

Столбцы с пропусками:
Age         177
Cabin       687
Embarked      2
dtype: int64


## Обработка пропущенных значений

In [4]:
df_clean = df.copy()

In [5]:
age_median = df_clean["Age"].median()
cabin_mode = df_clean["Cabin"].mode()[0]
embarked_mode = df_clean["Embarked"].mode()[0]

print("Медиана возраста:", age_median)
print("Мода каюты:", cabin_mode)
print("Мода порта посадки:", embarked_mode)

Медиана возраста: 28.0
Мода каюты: B96 B98
Мода порта посадки: S


In [6]:
df_clean["Age"] = df_clean["Age"].fillna(age_median)
df_clean["Cabin"] = df_clean["Cabin"].fillna(cabin_mode)
df_clean["Embarked"] = df_clean["Embarked"].fillna(embarked_mode)

In [7]:
print("Пропуски после заполнения:")
print(df_clean.isnull().sum())

print("\nОбщее количество пропусков:", df_clean.isnull().sum().sum())

Пропуски после заполнения:
PassengerId    0
Survived       0
Pclass         0
Name           0
Sex            0
Age            0
SibSp          0
Parch          0
Ticket         0
Fare           0
Cabin          0
Embarked       0
dtype: int64

Общее количество пропусков: 0


## Нормализация числовых признаков

In [8]:
df_processed = df_clean.copy()


In [9]:
numeric_columns = ["Age", "Fare", "SibSp", "Parch"]

df_processed[numeric_columns].describe()

,Age,Fare,SibSp,Parch
count,891.000000,891.000000,891.000000,891.000000
mean,29.361582,32.204208,0.523008,0.381594
std,13.019697,49.693429,1.102743,0.806057
min,0.420000,0.000000,0.000000,0.000000
25%,22.000000,7.910400,0.000000,0.000000
50%,28.000000,14.454200,0.000000,0.000000
75%,35.000000,31.000000,1.000000,0.000000
max,80.000000,512.329200,8.000000,6.000000


In [10]:
from sklearn.preprocessing import MinMaxScaler

scaler = MinMaxScaler()

df_processed[numeric_columns] = scaler.fit_transform(
    df_processed[numeric_columns]
)

In [11]:
print("Минимальные значения после нормализации:")
print(df_processed[numeric_columns].min())

print("\nМаксимальные значения после нормализации:")
print(df_processed[numeric_columns].max())

Минимальные значения после нормализации:
Age      0.0
Fare     0.0
SibSp    0.0
Parch    0.0
dtype: float64

Максимальные значения после нормализации:
Age      1.0
Fare     1.0
SibSp    1.0
Parch    1.0
dtype: float64


In [12]:
df_processed[numeric_columns].head()

,Age,Fare,SibSp,Parch
0,0.271174,0.014151,0.125,0.0
1,0.472229,0.139136,0.125,0.0
2,0.321438,0.015469,0.000,0.0
3,0.434531,0.103644,0.125,0.0
4,0.434531,0.015713,0.000,0.0


## Преобразование категориальных признаков

In [13]:
columns_to_drop = ["Name", "Ticket", "Cabin"]

df_model = df_processed.drop(columns=columns_to_drop)

print("Удалённые столбцы:", columns_to_drop)
print("Размер таблицы:", df_model.shape)

Удалённые столбцы: ['Name', 'Ticket', 'Cabin']
Размер таблицы: (891, 9)


In [14]:
categorical_columns = ["Sex", "Embarked", "Pclass"]

df_encoded = pd.get_dummies(
    df_model,
    columns=categorical_columns,
    drop_first=True,
    dtype=int
)

df_encoded.head()

,PassengerId,Survived,Age,SibSp,Parch,Fare,Sex_male,Embarked_Q,Embarked_S,Pclass_2,Pclass_3
0,1,0,0.271174,0.125,0.0,0.014151,1,0,1,0,1
1,2,1,0.472229,0.125,0.0,0.139136,0,0,0,0,0
2,3,1,0.321438,0.000,0.0,0.015469,0,0,1,0,1
3,4,1,0.434531,0.125,0.0,0.103644,0,0,1,0,0
4,5,0,0.434531,0.000,0.0,0.015713,1,0,1,0,1


In [15]:
text_columns = df_encoded.select_dtypes(include="object").columns.tolist()

print("Оставшиеся текстовые столбцы:", text_columns)
print("Итоговый размер таблицы:", df_encoded.shape)

Оставшиеся текстовые столбцы: []
Итоговый размер таблицы: (891, 11)


## Сохранение обработанных данных

In [16]:
output_path = "data/processed/processed_titanic.csv"

df_encoded.to_csv(output_path, index=False)

print("Обработанный датасет сохранён:")
print(output_path)

Обработанный датасет сохранён:
data/processed/processed_titanic.csv


In [17]:
df_check = pd.read_csv(output_path)

print("Размер сохранённого датасета:", df_check.shape)
print("Количество пропусков:", df_check.isnull().sum().sum())
print("Количество текстовых столбцов:",
      len(df_check.select_dtypes(include="object").columns))

df_check.head()

Размер сохранённого датасета: (891, 11)
Количество пропусков: 0
Количество текстовых столбцов: 0


,PassengerId,Survived,Age,SibSp,Parch,Fare,Sex_male,Embarked_Q,Embarked_S,Pclass_2,Pclass_3
0,1,0,0.271174,0.125,0.0,0.014151,1,0,1,0,1
1,2,1,0.472229,0.125,0.0,0.139136,0,0,0,0,0
2,3,1,0.321438,0.000,0.0,0.015469,0,0,1,0,1
3,4,1,0.434531,0.125,0.0,0.103644,0,0,1,0,0
4,5,0,0.434531,0.000,0.0,0.015713,1,0,1,0,1


## Вывод

В ходе лабораторной работы был загружен и исследован датасет Titanic.

Были обнаружены пропущенные значения в столбцах Age, Cabin и Embarked.
Пропуски были заполнены медианой и наиболее часто встречающимися значениями.

Числовые признаки Age, Fare, SibSp и Parch были нормализованы методом
MinMaxScaler. Категориальные признаки Sex, Embarked и Pclass были
преобразованы методом one-hot encoding.

Столбцы Name, Ticket и Cabin были удалены, поскольку содержат большое
количество уникальных текстовых значений. Итоговый датасет не содержит
пропусков и текстовых признаков и сохранён в файле
data/processed/processed_titanic.csv.